# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook guides you through loading and exploring the FAIR^2 dataset package using the `mlcroissant` library.

### Dataset Source
The dataset source is defined by a Croissant schema available at:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
List available record sets (`RecordSet`), along with their fields, IDs, and provide a brief summary.

In [ ]:
# Find and list all record sets by @id
print("Available record sets in this dataset:")
record_sets = list(dataset.record_sets)
for rs in record_sets:
    print(f"- {rs['@id']} : {rs.get('name', 'No name available')}")
    # List fields/columns for each record set
    if 'field' in rs:
        if isinstance(rs['field'], list):
            print("  Fields:")
            for f in rs['field']:
                if isinstance(f, dict):
                    print(f"    - {f.get('@id', '<unknown>')} : {f.get('name', '<no name>')}")
                else:
                    print(f"    - {f}")
        else:
            f = rs['field']
            if isinstance(f, dict):
                print(f"  Field: {f.get('@id', '<unknown>')} : {f.get('name', '<no name>')}")
            else:
                print(f"  Field: {f}")
    elif 'column' in rs:  # some schemas might use 'column' instead
        if isinstance(rs['column'], list):
            print("  Columns:")
            for col in rs['column']:
                print(f"    - {col.get('@id', '<unknown>')} : {col.get('name', '<no name>')}")
        else:
            print(f"  Column: {rs['column'].get('@id', '<unknown>')} : {rs['column'].get('name', '<no name>')}")
    else:
        print("  No fields/columns listed.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All record set and field references are based on their Croissant `@id`s.

In [ ]:
# Collect record set @id's for extraction
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

# Loop: Extract data for each record set (if any)
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

if dataframes:
    print(f"Loaded DataFrames for record sets: {list(dataframes.keys())}\n")
    # Pick the first record set for demonstration:
    selected_rs = record_set_ids[0]
    print(f"Columns in record set {selected_rs}:")
    print(dataframes[selected_rs].columns.tolist())
    display(dataframes[selected_rs].head())
else:
    print("No record sets with data detected in this dataset.")

## 4. Exploratory Data Analysis (EDA)
Apply data processing operations such as filtering numeric fields, normalization, and grouping. All references to data fields and grouping columns are made by their Croissant `@id`.

In [ ]:
import numpy as np
# Proceed only if a record set / DataFrame is available
if dataframes:
    df = dataframes[selected_rs]
    # Try to select a numeric field based on DataFrame dtypes
    numeric_field_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if numeric_field_candidates:
        numeric_field_id = numeric_field_candidates[0]
        print(f"Using numeric field for filtering: {numeric_field_id}")
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].notnull().any() else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std(ddof=0)
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try to find a non-numeric/grouping field
        group_field_candidates = [col for col in df.columns if col != numeric_field_id and (pd.api.types.is_object_dtype(df[col]) or pd.api.types.is_categorical_dtype(df[col]))]
        if group_field_candidates:
            group_field_id = group_field_candidates[0]
            print(f"Grouping by {group_field_id} and calculating the mean:")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(grouped_df.head())
        else:
            print("No suitable group field found to demonstrate grouping.")
    else:
        print("No numeric fields detected to perform EDA.")
else:
    print("No dataframes available to perform EDA.")

## 5. Visualization
Visualize data distributions or relationships using matplotlib and seaborn. Example below uses the first numeric and first non-numeric field, by their `@id`, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field_candidates:
    fig, axs = plt.subplots(1, 2, figsize=(14, 5))

    # Histogram of the numeric field
    sns.histplot(df[numeric_field_id].dropna(), kde=True, ax=axs[0], color='tab:blue')
    axs[0].set_title(f"Distribution of {numeric_field_id}")

    # Boxplot by group if possible
    if group_field_candidates:
        sns.boxplot(y=df[numeric_field_id], x=df[group_field_id], ax=axs[1])
        axs[1].set_title(f"{numeric_field_id} by {group_field_id}")
        axs[1].set_xticklabels(axs[1].get_xticklabels(), rotation=45)
    else:
        axs[1].axis('off')

    plt.tight_layout()
    plt.show()
else:
    print("No suitable fields/columns found for visualization.")

## 6. Conclusion
In this notebook, you have:
- Loaded the FAIR^2 Croissant dataset and reviewed its metadata using `mlcroissant`.
- Explored available record sets, fields, and column structure (using `@id` for robust referencing).
- Extracted data for each record set into pandas DataFrames for flexible downstream analysis.
- Performed basic exploratory data analysis: filtering, normalization, and grouping on numeric fields.
- Visualized numeric distributions and relationships between key variables.

This workflow demonstrates how Croissant datasets can be programmatically explored and processed—enabling reproducible, FAIR data science!